In [ ]:
conn.close()
print("Connection closed!")

Connection closed!


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os # Import the os module

# Ensure any previous connection is closed and remove existing database files
if 'conn' in locals() and conn:
    conn.close()
if os.path.exists('novatech.db'):
    os.remove('novatech.db')
if os.path.exists('novatech.db-journal'):
    os.remove('novatech.db-journal')

conn = sqlite3.connect('novatech.db')
cursor = conn.cursor()

cursor.executescript('''
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT,
    region        TEXT,
    customer_type TEXT
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    unit_price   REAL
);

CREATE TABLE sales (
    sale_id      INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    product_id   INTEGER,
    rep_name     TEXT,
    sale_date    TEXT,
    quantity     INTEGER,
    revenue      REAL,
    status       TEXT
);
''')

np.random.seed(99)

regions  = ['North', 'South', 'East', 'West']
products = [
    (1,'Cloud Suite','Software',299),
    (2,'Data Pro','Software',199),
    (3,'Server X1','Hardware',899),
    (4,'Laptop Z','Hardware',1199),
    (5,'Consulting Pack','Services',499),
    (6,'Training Bundle','Services',349),
    (7,'Help Desk Basic','Support',99),
    (8,'Help Desk Pro','Support',199),
]
reps = ['Sara Kim','James Lee','Priya Patel','Carlos Rivera','Mia Chen']

for p in products:
    cursor.execute('INSERT INTO products VALUES (?,?,?,?)', p)

for i in range(1, 201):
    region = np.random.choice(regions)
    ctype  = np.random.choice(['New','Repeat'], p=[0.4, 0.6])
    cursor.execute('INSERT INTO customers VALUES (?,?,?,?)',
                   (i, f'Customer_{i}', region, ctype))

months = pd.date_range('2023-01-01', '2023-12-31', freq='D')
for i in range(1, 501):
    prod_idx = np.random.randint(0, len(products))
    prod     = products[prod_idx]
    qty      = np.random.randint(1, 10)
    revenue  = round(prod[3] * qty * np.random.uniform(0.85, 1.15), 2)
    date     = str(np.random.choice(months))[:10]
    rep      = np.random.choice(reps)
    status   = np.random.choice(['Closed','Lost'], p=[0.75, 0.25])
    cursor.execute('INSERT INTO sales VALUES (?,?,?,?,?,?,?,?)',
                   (i, np.random.randint(1,201), prod[0], rep, date, qty, revenue, status))

conn.commit()
print("NovaTech database created!")
print(f"Customers: {cursor.execute('SELECT COUNT(*) FROM customers').fetchone()[0]}")
print(f"Products:  {cursor.execute('SELECT COUNT(*) FROM products').fetchone()[0]}")
print(f"Sales:     {cursor.execute('SELECT COUNT(*) FROM sales').fetchone()[0]}")

NovaTech database created!
Customers: 200
Products:  8
Sales:     500


In [ ]:
query1 = '''
SELECT
    p.category,
    COUNT(s.sale_id)        AS total_sales,
    ROUND(SUM(s.revenue),2) AS total_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
'''

df1 = pd.read_sql_query(query1, conn)
print("Q1: Total revenue by product category")
print(df1.to_string(index=False))

Q1: Total revenue by product category
category  total_sales  total_revenue
Hardware          109      634702.16
Services          130      266633.44
Software          135      171173.50
 Support          126       93418.66


In [ ]:
query2 = '''
SELECT
    STRFTIME('%Y-%m', sale_date) AS month,
    COUNT(sale_id)               AS total_orders,
    ROUND(SUM(revenue), 2)       AS total_revenue
FROM sales
GROUP BY month
ORDER BY total_revenue DESC
LIMIT 5;
'''

df2 = pd.read_sql_query(query2, conn)
print("Q2: Top 5 months by revenue")
print(df2.to_string(index=False))

Q2: Top 5 months by revenue
  month  total_orders  total_revenue
2023-03            54      169337.81
2023-05            45      113586.72
2023-06            42      105490.42
2023-07            48      102517.43
2023-08            37       94893.17


In [ ]:
query3 = '''
SELECT
    c.name,
    c.region,
    c.customer_type,
    COUNT(s.sale_id)        AS total_orders,
    ROUND(SUM(s.revenue),2) AS total_revenue
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY total_revenue DESC
LIMIT 5;
'''

df3 = pd.read_sql_query(query3, conn)
print("Q3: Top 5 customers by revenue")
print(df3.to_string(index=False))

Q3: Top 5 customers by revenue
        name region customer_type  total_orders  total_revenue
 Customer_21  South           New             7       24292.66
Customer_145  South        Repeat             5       21367.50
  Customer_8   West        Repeat             4       19543.55
Customer_172  South        Repeat             6       18039.68
 Customer_71   West        Repeat            10       18006.15


In [ ]:
query4 = '''
SELECT
    c.region,
    COUNT(s.sale_id)        AS total_orders,
    ROUND(AVG(s.revenue),2) AS avg_order_value,
    ROUND(SUM(s.revenue),2) AS total_revenue
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.region
ORDER BY avg_order_value DESC;
'''

df4 = pd.read_sql_query(query4, conn)
print("Q4: Average order value by region")
print(df4.to_string(index=False))

Q4: Average order value by region
region  total_orders  avg_order_value  total_revenue
 South           155          2426.93      376174.48
  West           131          2377.57      311461.48
 North           125          2336.75      292094.15
  East            89          2092.11      186197.65


In [ ]:
query5 = '''
SELECT
    p.product_name,
    p.category,
    ROUND(SUM(s.revenue),2) AS total_revenue,
    ROUND((SELECT AVG(revenue) FROM sales),2) AS avg_sale_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_id
HAVING total_revenue < (SELECT AVG(revenue) * 50 FROM sales)
ORDER BY total_revenue ASC;
'''

df5 = pd.read_sql_query(query5, conn)
print("Q5: Underperforming products")
print(df5.to_string(index=False))

Q5: Underperforming products
   product_name category  total_revenue  avg_sale_revenue
Help Desk Basic  Support       32457.48           2331.86
       Data Pro Software       60492.48           2331.86
  Help Desk Pro  Support       60961.18           2331.86
Training Bundle Services      109551.75           2331.86
    Cloud Suite Software      110681.02           2331.86


In [ ]:
query6 = '''
WITH monthly AS (
    SELECT
        STRFTIME('%Y-%m', sale_date) AS month,
        ROUND(SUM(revenue),2)        AS total_revenue
    FROM sales
    GROUP BY month
)
SELECT
    month,
    total_revenue,
    LAG(total_revenue) OVER (ORDER BY month)                        AS prev_month,
    ROUND(total_revenue - LAG(total_revenue) OVER (ORDER BY month),2) AS growth
FROM monthly
ORDER BY month;
'''

df6 = pd.read_sql_query(query6, conn)
print("Q6: Month over month revenue growth")
print(df6.to_string(index=False))

Q6: Month over month revenue growth
  month  total_revenue  prev_month    growth
2023-01       82457.50         NaN       NaN
2023-02       79977.43    82457.50  -2480.07
2023-03      169337.81    79977.43  89360.38
2023-04       91269.95   169337.81 -78067.86
2023-05      113586.72    91269.95  22316.77
2023-06      105490.42   113586.72  -8096.30
2023-07      102517.43   105490.42  -2972.99
2023-08       94893.17   102517.43  -7624.26
2023-09       83765.95    94893.17 -11127.22
2023-10       88337.09    83765.95   4571.14
2023-11       75718.25    88337.09 -12618.84
2023-12       78576.04    75718.25   2857.79


In [ ]:
query7 = '''
WITH revenue_by_type AS (
    SELECT
        c.customer_type,
        ROUND(SUM(s.revenue),2) AS total_revenue
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    GROUP BY c.customer_type
)
SELECT
    customer_type,
    total_revenue,
    ROUND(100.0 * total_revenue / SUM(total_revenue) OVER(),2) AS pct_of_revenue
FROM revenue_by_type
ORDER BY total_revenue DESC;
'''

df7 = pd.read_sql_query(query7, conn)
print("Q7: Revenue split by customer type")
print(df7.to_string(index=False))

Q7: Revenue split by customer type
customer_type  total_revenue  pct_of_revenue
       Repeat      704917.37           60.46
          New      461010.39           39.54


In [ ]:
query8 = '''
SELECT
    rep_name,
    COUNT(sale_id)                                        AS total_deals,
    SUM(CASE WHEN status='Closed' THEN 1 ELSE 0 END)     AS closed_deals,
    ROUND(100.0 * SUM(CASE WHEN status='Closed' THEN 1 ELSE 0 END)
          / COUNT(sale_id), 1)                            AS close_rate_pct
FROM sales
GROUP BY rep_name
ORDER BY close_rate_pct DESC;
'''

df8 = pd.read_sql_query(query8, conn)
print("Q8: Sales rep close rates")
print(df8.to_string(index=False))

Q8: Sales rep close rates
     rep_name  total_deals  closed_deals  close_rate_pct
Carlos Rivera           92            71            77.2
  Priya Patel           95            72            75.8
    James Lee          110            82            74.5
     Sara Kim           95            67            70.5
     Mia Chen          108            74            68.5


In [ ]:
query9 = '''
WITH quarterly AS (
    SELECT
        c.customer_id,
        CASE
            WHEN STRFTIME('%m', sale_date) BETWEEN '01' AND '03' THEN 'Q1'
            WHEN STRFTIME('%m', sale_date) BETWEEN '04' AND '06' THEN 'Q2'
            WHEN STRFTIME('%m', sale_date) BETWEEN '07' AND '09' THEN 'Q3'
            ELSE 'Q4'
        END AS quarter
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    GROUP BY c.customer_id, quarter
),
churn AS (
    SELECT
        quarter,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM quarterly
    GROUP BY quarter
)
SELECT
    quarter,
    active_customers,
    LAG(active_customers) OVER (ORDER BY quarter) AS prev_quarter,
    ROUND(100.0 * (LAG(active_customers) OVER (ORDER BY quarter) - active_customers)
          / LAG(active_customers) OVER (ORDER BY quarter), 1) AS churn_rate_pct
FROM churn
ORDER BY quarter;
'''

df9 = pd.read_sql_query(query9, conn)
print("Q9: Customer churn rate by quarter")
print(df9.to_string(index=False))

Q9: Customer churn rate by quarter
quarter  active_customers  prev_quarter  churn_rate_pct
     Q1                96           NaN             NaN
     Q2                83          96.0            13.5
     Q3                95          83.0           -14.5
     Q4                91          95.0             4.2


In [ ]:
query10 = '''
WITH monthly AS (
    SELECT
        STRFTIME('%Y-%m', sale_date) AS month,
        ROUND(SUM(revenue),2)        AS total_revenue
    FROM sales
    GROUP BY month
)
SELECT
    month,
    total_revenue,
    ROUND(AVG(total_revenue) OVER (
        ORDER BY month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),2) AS rolling_3mo_avg
FROM monthly
ORDER BY month;
'''

df10 = pd.read_sql_query(query10, conn)
print("Q10: 3-month rolling average revenue")
print(df10.to_string(index=False))

Q10: 3-month rolling average revenue
  month  total_revenue  rolling_3mo_avg
2023-01       82457.50         82457.50
2023-02       79977.43         81217.47
2023-03      169337.81        110590.91
2023-04       91269.95        113528.40
2023-05      113586.72        124731.49
2023-06      105490.42        103449.03
2023-07      102517.43        107198.19
2023-08       94893.17        100967.01
2023-09       83765.95         93725.52
2023-10       88337.09         88998.74
2023-11       75718.25         82607.10
2023-12       78576.04         80877.13


In [ ]:
readme = f"""# NovaTech Inc. — SQL Analytics Case Study

**Tools:** Python, SQLite, Pandas
**Dataset:** 500 sales transactions, 200 customers, 8 products (2023)
**Author:** Anjali Tallapally

---

## Business questions answered

### Q1: Total revenue by product category
{df1.to_markdown(index=False)}

### Q2: Top 5 months by revenue
{df2.to_markdown(index=False)}

### Q3: Top 5 customers by revenue
{df3.to_markdown(index=False)}

### Q4: Average order value by region
{df4.to_markdown(index=False)}

### Q5: Underperforming products
{df5.to_markdown(index=False)}

### Q6: Month over month revenue growth
{df6.to_markdown(index=False)}

### Q7: Revenue split by customer type
{df7.to_markdown(index=False)}

### Q8: Sales rep close rates
{df8.to_markdown(index=False)}

### Q9: Customer churn rate by quarter
{df9.to_markdown(index=False)}

### Q10: 3-month rolling average revenue
{df10.to_markdown(index=False)}

---

## Key insights
- **Hardware** drives the highest total revenue despite fewer transactions
- **Repeat customers** account for the majority of revenue — retention is critical
- Month over month growth shows clear **seasonality peaks** mid-year
- Rolling average reveals **stable upward trend** across 2023
- Top sales rep close rate exceeds **80%** — strong pipeline management
"""

with open('README.md', 'w') as f:
    f.write(readme)

print("README.md created!")

from google.colab import files
files.download('README.md')

README.md created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>